# Oscar — 03: SVM (working)

Run `report/01-Data.ipynb` first.

Same features as the final LR (fixes 1-4 + ratios) and same 5 folds as `02`. Score to beat: LR CV AUC 0.7722.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

import prep                         # Oscar/prep.py: fixes 1-4 + ratio features

raw_train, y_train = prep.load("train")
X = prep.prepare(raw_train, ratios=True)                    # fixes 1-3 + ratios; scaling is done inside each fold
cv = StratifiedKFold(5, shuffle=True, random_state=2026)    # same folds as 02

print("X:", X.shape, f"  {y_train.mean():.2%} defaulted")

X: (17997, 61)   22.12% defaulted


## 1. First look

Same test bench as `02` section 3. SVC gives scores, not probabilities, so AUC only for now (no log loss).

In [2]:
results, fold_auc = {}, {}      # name -> CV summary; name -> AUC on each fold

def cv_model(name, model):
    """5-fold CV AUC; standardising is learned inside each fold."""
    r = cross_validate(make_pipeline(StandardScaler(), model), X, y_train, cv=cv,
                       scoring="roc_auc", return_train_score=True)
    fold_auc[name] = r["test_score"]
    results[name] = {"auc": r["test_score"].mean(), "auc_sd": r["test_score"].std(),
                     "train_auc": r["train_score"].mean(), "fit_s": r["fit_time"].mean()}
    return pd.DataFrame.from_dict(results, orient="index").round(4)

def compare(ref="LR"):
    """AUC gain over `ref`, fold by fold: every model sat the same 5 folds."""
    gain = pd.DataFrame(fold_auc).sub(fold_auc[ref], axis=0)
    return pd.DataFrame({"mean gain": gain.mean(), "folds better": (gain > 0).sum()}).round(4)

cv_model("LR", LogisticRegression(C=0.0316, l1_ratio=0, solver="liblinear", max_iter=1000))   # final LR from 02
cv_model("SVM rbf (defaults)", SVC())                                                        # C=1, gamma="scale"

,auc,auc_sd,train_auc,fit_s
LR,0.7722,0.0140,0.7775,0.0559
SVM rbf (defaults),0.7244,0.0112,0.8528,2.9200


In [3]:
compare()

,mean gain,folds better
LR,0.0000,0
SVM rbf (defaults),-0.0478,0
